# Predicting how often a family went without enough food

**Matthias Muscat · ICS5110 · Multinomial logistic regression**

The model predicts the answer to World Values Survey Q51 from six background questions encoded as 18 inputs. The classes are **0 never, 1 rarely, 2 sometimes and 3 often** in the previous 12 months. Multinomial logistic regression estimates a separate score for each class. The highest scoring class is the prediction.

This notebook is configured for the **real Wave 7 survey**. The survey run estimates performance for held-out respondents from the participating samples. [Private data access and rebuild instructions](../work/muscat/README.md) · [Review the survey results](../work/muscat/results/logistic_regression/README.md)

## Dataset and frozen test split

The member protocol uses an 80/20 respondent split stratified by Q51, seed 42. Sorting by the original source-row key before splitting makes the membership repeatable. An alternative shared assignment can be supplied through `SPLIT_CSV` with the columns `meta_source_row` and `split` (`train` or `test`). The private survey CSV must match its published checksum. To check the code without the survey, select `demo`. Only the 18 listed baseline inputs enter the model.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, make_scorer, precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

DATA_SOURCE = "survey"  # "survey" for the experiment, "demo" for fictional respondents
SPLIT_CSV = None        # optionally set a repository-relative path to a shared assignment

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "work/muscat/q51_dataset.json").is_file()), None
)
if root is None:
    raise FileNotFoundError("Open the notebook from the repository root or its 03_code directory.")
member = root / "work" / "muscat"
sys.path.insert(0, str(member))
from q51_model_data import CODES, DATASET, FEATURES, KEY, LABELS, TARGET
from q51_model_data import align_split, example_data, load_survey, proposed_split
from q51_performance import alert_counts, assignment_digest, choose_alert, export_model
from q51_performance import multiclass_scores, save_confusion_plot, save_json

if DATA_SOURCE == "survey":
    data = load_survey(member / "private" / DATASET["filename"])
    source_label = "WVS Wave 7 survey"
elif DATA_SOURCE == "demo":
    data = example_data()
    source_label = "Synthetic demonstration"
else:
    raise ValueError("Select survey or demo.")

data = data.sort_values(KEY).reset_index(drop=True)
assignments = proposed_split(data) if SPLIT_CSV is None else pd.read_csv(root / SPLIT_CSV)
partition = align_split(data, assignments)
split_hash = assignment_digest(assignments[[KEY, "split"]].sort_values(KEY))
train, test = partition.eq("train"), partition.eq("test")
X_train, y_train = data.loc[train, FEATURES], data.loc[train, TARGET]
counts = pd.crosstab(data[TARGET], partition).reindex(index=CODES, columns=["train", "test"], fill_value=0)
counts.index = LABELS
print(f"{source_label} | {len(data):,} respondents | {len(FEATURES)} predictor columns")
print(counts.to_string())

WVS Wave 7 survey | 96,704 respondents | 18 predictor columns
split      train   test
Never      53418  13354
Rarely     12033   3008
Sometimes   8746   2187
Often       3166    792


## Choose settings inside the training set

Five shuffled, stratified folds use seed 42. Each training respondent is in one validation fold. Each candidate fits a new scaler and model within its training fold. Candidate regularisation values are `C = 0.01, 0.1, 1, 10`. They are compared with ordinary fitting and inverse-frequency class weighting. The latter gives rarer training classes more weight. Both are evaluated with **unweighted macro-F1**, the average of the four class F1 scores.

The candidate with the highest mean validation macro-F1 is refit on all training respondents. Exact ties follow the declared order, with ordinary weighting first and smaller C first within each weighting. Survey design weights are reserved for a separate sensitivity check.

In [2]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_train, y_train))
fold_id = np.zeros(len(y_train), dtype=int)
fold_counts = []
for number, (_, valid) in enumerate(folds, start=1):
    fold_id[valid] = number
    fold_counts.append(y_train.iloc[valid].value_counts().reindex(CODES, fill_value=0).astype(int).tolist())
fold_hash = assignment_digest(pd.DataFrame({KEY: data.loc[train, KEY].to_numpy(), "fold": fold_id}))

pipeline = make_pipeline(StandardScaler(), LogisticRegression(solver="lbfgs", max_iter=1000))
grid = [
    {"logisticregression__C": [0.01, 0.1, 1.0, 10.0], "logisticregression__class_weight": [weights]}
    for weights in [None, "balanced"]
]
macro_f1 = make_scorer(f1_score, labels=CODES, average="macro", zero_division=0)
search = GridSearchCV(pipeline, grid, cv=folds, scoring=macro_f1, n_jobs=1, refit=True, error_score="raise")
with warnings.catch_warnings(), threadpool_limits(limits=1):
    warnings.simplefilter("error", ConvergenceWarning)
    search.fit(X_train, y_train)
    model = search.best_estimator_
    oof_probabilities = cross_val_predict(model, X_train, y_train, cv=folds, method="predict_proba", n_jobs=1)

cv_rows = []
for index, params in enumerate(search.cv_results_["params"]):
    cv_rows.append({
        "C": float(params["logisticregression__C"]), "class_weight": params["logisticregression__class_weight"],
        "mean_macro_f1": float(search.cv_results_["mean_test_score"][index]),
        "sd_macro_f1": float(search.cv_results_["std_test_score"][index]),
        "fold_macro_f1": [float(search.cv_results_[f"split{k}_test_score"][index]) for k in range(len(folds))],
    })
selected = cv_rows[search.best_index_]
print(f"{source_label} | five-fold validation, training respondents only")
print(pd.DataFrame(cv_rows)[["C", "class_weight", "mean_macro_f1", "sd_macro_f1"]]
      .fillna({"class_weight": "ordinary"}).round(5).to_string(index=False))
print(f"Selected C={selected['C']:g}, class weighting={selected['class_weight'] or 'ordinary'}.")

WVS Wave 7 survey | five-fold validation, training respondents only
    C class_weight  mean_macro_f1  sd_macro_f1
 0.01     ordinary        0.23124      0.00250
 0.10     ordinary        0.23187      0.00274
 1.00     ordinary        0.23196      0.00270
10.00     ordinary        0.23196      0.00270
 0.01     balanced        0.30505      0.00333
 0.10     balanced        0.30521      0.00336
 1.00     balanced        0.30523      0.00331
10.00     balanced        0.30524      0.00330
Selected C=10, class weighting=balanced.


## Choose a recurring-shortage alert before opening the test set

For a separate yes/no alert, add the model scores for “sometimes” and “often”. A missed recurring shortage is assigned two illustrative cost units and a false alarm one. The ratio represents a scenario for comparing decisions. It is not an observed social cost. We compare cutoffs from 0 to 1 in steps of 0.05 using only out-of-fold training scores. If costs tie, the higher cutoff is selected. The alert leaves the original four-class prediction intact. Class-weighted fits can change the scale of scores, so the cutoff has to be checked with data.

In [3]:
if model.classes_.tolist() != CODES:
    raise ValueError("Unexpected order for the four model classes.")
oof_scores = oof_probabilities[:, [2, 3]].sum(axis=1)
alert_validation, chosen_cutoff = choose_alert(y_train, oof_scores)
print(f"{source_label} | alert cutoff selected from training predictions: {chosen_cutoff:.2f}")
print("The final model settings and alert cutoff are fixed before test scoring.")

WVS Wave 7 survey | alert cutoff selected from training predictions: 0.65
The final model settings and alert cutoff are fixed before test scoring.


## Evaluate the fixed model on the held-out respondents

Macro-F1 gives equal importance to each observed answer. Accuracy counts all correct predictions. Mean absolute class error measures the distance in answer categories between the prediction and the reported answer. A “never” prediction for an “often” response has a distance of three. These distances do not measure how many meals were missed. The comparison model always predicts the most common training answer.

In [4]:
X_test, y_test = data.loc[test, FEATURES], data.loc[test, TARGET]
probabilities = model.predict_proba(X_test)
if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1):
    raise ValueError("Invalid four-class model scores.")
predicted = model.classes_[probabilities.argmax(axis=1)]
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_predicted = baseline.predict(X_test)
test_scores = {
    "most_frequent": multiclass_scores(y_test, baseline_predicted),
    "logistic_regression": multiclass_scores(y_test, predicted),
}
survey_weights = data.loc[test, ["meta_W_WEIGHT", "meta_S018"]].astype(float).prod(axis=1)
weighted_scores = {
    "most_frequent": multiclass_scores(y_test, baseline_predicted, survey_weights),
    "logistic_regression": multiclass_scores(y_test, predicted, survey_weights),
}
print(f"{source_label} | unweighted final test results, n = {len(y_test):,}")
print(pd.DataFrame(test_scores).T.round(4).to_string())
print("\nSensitivity using the national weight times the equal-survey-size factor")
print(pd.DataFrame(weighted_scores).T.round(4).to_string())

WVS Wave 7 survey | unweighted final test results, n = 19,341
                     accuracy  macro_f1  macro_recall  mean_absolute_class_error
most_frequent          0.6905    0.2042        0.2500                     0.5045
logistic_regression    0.4925    0.3043        0.3655                     0.8766

Sensitivity using the national weight times the equal-survey-size factor
                     accuracy  macro_f1  macro_recall  mean_absolute_class_error
most_frequent          0.6862    0.2035        0.2500                     0.5115
logistic_regression    0.4763    0.2988        0.3645                     0.9079


In [5]:
matrix = confusion_matrix(y_test, predicted, labels=CODES)
precision, recall, f1, support = precision_recall_fscore_support(y_test, predicted, labels=CODES, zero_division=0)
per_class = pd.DataFrame({
    "label": LABELS, "test_support": support, "correct": matrix.diagonal(), "predicted_count": matrix.sum(axis=0),
    "precision": precision, "recall": recall, "f1": f1,
})
print(f"{source_label} | performance by answer (undefined class metrics are set to zero)")
print(per_class.round(4).to_string(index=False))
print("\nConfusion matrix (actual answers in rows, model predictions in columns)")
print(pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_string())

WVS Wave 7 survey | performance by answer (undefined class metrics are set to zero)
    label  test_support  correct  predicted_count  precision  recall     f1
    Never         13354     8190             9971     0.8214  0.6133 0.7023
   Rarely          3008      596             3032     0.1966  0.1981 0.1974
Sometimes          2187      351             2137     0.1642  0.1605 0.1623
    Often           792      388             4201     0.0924  0.4899 0.1554

Confusion matrix (actual answers in rows, model predictions in columns)
           Never  Rarely  Sometimes  Often
Never       8190    1982       1169   2013
Rarely      1018     596        503    891
Sometimes    562     365        351    909
Often        201      89        114    388


## How the alert changes misses and false alarms

The table reports the cutoff chosen from training and the prespecified comparison cutoffs 0.25, 0.50 and 0.75. For respondents aged 65 or over with a recorded age, it also compares the selected cutoff with 0.50. This subgroup was chosen before test evaluation. The survey weight sensitivity above, the training class weights and the illustrative costs here represent different choices.

In [6]:
recurring_scores = probabilities[:, [2, 3]].sum(axis=1)
alert_test = [alert_counts(y_test, recurring_scores, t) for t in sorted({0.25, 0.50, 0.75, chosen_cutoff})]
older = (X_test["x_age"].ge(65) & X_test["x_age_missing"].eq(0)).to_numpy()
alert_older = [alert_counts(y_test.to_numpy()[older], recurring_scores[older], t)
               for t in sorted({0.50, chosen_cutoff})]
print(f"{source_label} | held-out recurring-shortage alert (chosen cutoff {chosen_cutoff:.2f})")
print(pd.DataFrame(alert_test).to_string(index=False))
print("\nRespondents aged 65 or over")
print(pd.DataFrame(alert_older).to_string(index=False))

WVS Wave 7 survey | held-out recurring-shortage alert (chosen cutoff 0.65)
 threshold  respondents  hits  misses  false_alarms  correct_negatives  illustrative_cost
      0.25        19341  2880      99         14160               2202              14358
      0.50        19341  1903    1076          5247              11115               7399
      0.65        19341   772    2207          1124              15238               5538
      0.75        19341   146    2833           131              16231               5797

Respondents aged 65 or over
 threshold  respondents  hits  misses  false_alarms  correct_negatives  illustrative_cost
      0.50         2390   114     119           328               1829                566
      0.65         2390    35     198            53               2104                449


## Saved results

The four-class metrics, validation scores, survey-weight sensitivity, confusion counts and alert results are exported as aggregate JSON. A second JSON file contains the fitted scaler and logistic coefficients in feature order. To score a locally encoded record, standardise its 18 values, apply the coefficients and intercepts, then take the softmax. A check below reproduces the held-out scores from the exported parameters. Individual records and predictions are kept out of the public results.

In [7]:
import json
import platform
from tempfile import TemporaryDirectory
from scipy.special import softmax

if DATA_SOURCE == "survey":
    if SPLIT_CSV is None:
        result_dir = member / "results" / "logistic_regression"
    else:
        result_dir = member / "private" / "logistic_regression_custom_split"
    result_dir.mkdir(parents=True, exist_ok=True)
    artifact = export_model(model, DATASET["sha256"], split_hash)
    released = json.loads(json.dumps(artifact, allow_nan=False))
    scaled = (X_test[released["features"]].to_numpy() - np.array(released["scaler_mean"])) / np.array(released["scaler_scale"])
    logits = scaled @ np.array(released["coefficients"]).T + np.array(released["intercepts"])
    if not np.allclose(softmax(logits, axis=1), probabilities, atol=1e-12, rtol=0):
        raise ValueError("Exported parameters differ from the fitted estimator.")
    result = {
        "dataset": {"name": DATASET["name"], "sha256": DATASET["sha256"], "rows": len(data)},
        "split": {"method": "stratified 80/20, seed 42" if SPLIT_CSV is None else "provided assignment CSV",
                  "sha256": split_hash, "train_n": int(train.sum()), "test_n": int(test.sum())},
        "class_order": LABELS, "train_by_class": counts["train"].tolist(), "test_by_class": counts["test"].tolist(),
        "validation": {"method": "stratified 5-fold, shuffled, seed 42", "sha256": fold_hash,
                       "fold_class_support": fold_counts, "selection_metric": "mean unweighted macro-F1",
                       "candidates": cv_rows, "selected": selected},
        "test": {"scores": test_scores, "survey_weighted_scores": weighted_scores,
                 "per_class": per_class.to_dict("records"), "confusion_actual_by_predicted": matrix.tolist()},
        "recurring_alert": {"classes": [2, 3], "miss_cost": 2, "false_alarm_cost": 1,
                            "chosen_threshold": chosen_cutoff, "training_oof": alert_validation,
                            "test": alert_test, "test_age_65_plus": alert_older},
        "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                     "scikit_learn": sklearn.__version__},
    }
    with TemporaryDirectory(dir=result_dir) as temporary:
        staging = Path(temporary)
        save_json(staging / "model.json", artifact)
        save_json(staging / "metrics.json", result)
        save_confusion_plot(matrix, staging / "confusion_matrix.png")
        for path in staging.iterdir():
            path.replace(result_dir / path.name)
    print(f"Saved aggregate results, confusion matrix and verified fitted model to {result_dir.relative_to(root)}.")
else:
    print("Synthetic demonstration complete.")

Saved aggregate results, confusion matrix and verified fitted model to work/muscat/results/logistic_regression.


## Sources

- Haerpfer et al. (2022), *World Values Survey Wave 7 Cross-National Data-Set*, v6.0.0. [Dataset and documentation](https://doi.org/10.14281/18241.24)
- Scikit-learn 1.8.0. [Logistic regression](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.LogisticRegression.html), [cross-validation](https://scikit-learn.org/1.8/modules/cross_validation.html) and [parameter search](https://scikit-learn.org/1.8/modules/generated/sklearn.model_selection.GridSearchCV.html).
- Matthias Muscat’s [decision and AI-use record](../work/muscat/log.md).